# Frontier LLM Research Engineering — Phase 0 scaled pilots

Run on Colab with a GPU runtime (**A100** preferred; **L4** for the FP8 pilot; T4 for the free-GPU
checks). Run the cells top to bottom. Every pilot writes to Google Drive and resumes if the session
disconnects: just run the same cell again.

Order matters when compute units run out: **P1 (noise floor) first**, then the others.

In [ ]:
# 1. Mount Drive (outputs survive disconnects) and choose where pilots write.
from google.colab import drive
drive.mount('/content/drive')
PILOT_DIR = '/content/drive/MyDrive/frontier-llm-pilots'
import os; os.makedirs(PILOT_DIR, exist_ok=True); os.environ['PILOT_DIR'] = PILOT_DIR
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2. Get the course code. Option A: the GitHub repo (set REPO_URL; a private repo needs a token in the URL).
#    Option B: upload a zip of the repo to Drive and set ZIP_PATH.
REPO_URL = 'https://github.com/tal-giladi/frontier-llm-research-course.git'
ZIP_PATH = ''   # e.g. '/content/drive/MyDrive/frontier-llm-research-course.zip'
import os
if ZIP_PATH:
    !rm -rf /content/course && mkdir -p /content/course && unzip -q "$ZIP_PATH" -d /content/course
    ROOT = '/content/course/' + os.listdir('/content/course')[0] if len(os.listdir('/content/course')) == 1 else '/content/course'
else:
    !rm -rf /content/course && git clone -q "$REPO_URL" /content/course
    ROOT = '/content/course'
%cd $ROOT
!git log --oneline -1 2>/dev/null || true

In [ ]:
# 3. Install the pinned stack (references/versions.md). PyTorch 2.14.1 CUDA wheel, then the lab package.
!pip -q install torch==2.14.1 --index-url https://download.pytorch.org/whl/cu128
!pip -q install numpy==2.5.3 scipy transformers==5.18.0 safetensors tokenizers==0.23.2 datasets==5.0.1 matplotlib==3.11.2 pytest==9.1.1 pyyaml==6.0.3 torchao==0.18.0
!pip -q install -e labs/common
import torch; print(torch.__version__, torch.cuda.get_device_name(0))
!python -m pytest labs/common/tests/test_model.py -x -q 2>&1 | tail -3   # quick check; the full suite runs on CPU in the build

In [ ]:
# 4. Data-v0 at pilot size (vocab 32768). ~800k documents is ~0.8B train tokens: enough for the P1 ladder.
#    Writes to Drive; skipped if already there. Expect roughly 30-60 minutes and ~2 GB.
DATA = f'{PILOT_DIR}/data-v0-32k'
import os
if not os.path.exists(f'{DATA}/meta.json'):
    !python -m frontierlab.data.prepare --docs 800000 --vocab 32768 --out "$DATA"
# Lab data and every runs/ folder live on Drive, so a disconnect or a new session loses nothing.
!mkdir -p "$PILOT_DIR/labdata" "$PILOT_DIR/runs" && rm -rf labs/common/data runs
!ln -sfn "$PILOT_DIR/labdata" labs/common/data && ln -sfn "$PILOT_DIR/runs" runs
!rm -rf labs/common/data/v0 && ln -sfn "$DATA" labs/common/data/v0
!cat "$DATA/meta.json" | head -40

## P16 — Module 16: agent environments and misspecified rewards (T4 variants)

The 16.1 verifier audit, then seed 0 of each 16.4 arm (misspecified rewards, detection) on Qwen3-0.6B-Base. The main path (~11–19 GPU-h) stays PROJECTED.

In [ ]:
O16 = f'{PILOT_DIR}/p16'; os.makedirs(O16, exist_ok=True)
!python curriculum/pilots/once.py P16 0 'python labs/module-16/lesson-01/verifier_lab.py 2>&1 | tee "$O16/16-1-verifiers.txt" | tail -15'
# The t4 variant prints one command per run; run seed 0 of each arm (6 runs of 150 steps).
cmds = !python labs/module-16/lesson-04/hacking_lab.py --variant t4
for c in [c for c in cmds if '--seed 0' in c]:
    !{c} 2>&1 | tail -4
!python curriculum/pilots/once.py P16 1 'python labs/module-16/lesson-04/hacking_lab.py --part traces 2>&1 | tee "$O16/16-4-traces.txt" | tail -10'

In [ ]:
# Last cell: marks P16 complete when no command of it has failed; the status notebook reads this.
import glob, os
failed = glob.glob(f'{PILOT_DIR}/.failed/P16-*')
if failed:
    print('P16 NOT complete; failed commands (rerun this notebook to retry):', failed)
else:
    open(f'{PILOT_DIR}/.done/P16-COMPLETE', 'w').write('ok')
    print('P16 complete')